# Занятие 5. Pandas: преобразования, строки, даты, группировки и объединение таблиц

На прошлом занятии мы научились:

- загружать CSV;
- смотреть структуру `DataFrame`;
- выбирать строки и столбцы;
- использовать `loc` / `iloc`;
- фильтровать и сортировать данные.

Теперь переходим от **чтения таблицы** к её **преобразованию и анализу**.

Главная линия занятия:

```text
исходные столбцы
→ новые признаки
→ строки и даты
→ groupby
→ merge
→ анализ объединённых данных
```


## Что нужно уметь после занятия

- создавать новые столбцы;
- выполнять векторные операции со столбцами;
- использовать строковые методы через `.str`;
- преобразовывать строки в даты через `pd.to_datetime`;
- извлекать части даты через `.dt`;
- группировать данные через `groupby`;
- считать агрегаты по группам;
- понимать ключ объединения;
- объединять таблицы через `merge`;
- различать `inner` и `left` на базовом уровне.


# Часть 1. Новые столбцы

## 1. Загружаем данные


In [ ]:
import pandas as pd

orders = pd.read_csv("lesson_05_orders.csv")
customers = pd.read_csv("lesson_05_customers.csv")

orders.head()


Посмотрим на структуру:


In [ ]:
orders.info()


В таблице есть:

- `quantity` — количество;
- `unit_price` — цена одной единицы.

Полную стоимость заказа можно получить как:

```text
quantity × unit_price
```


## 2. Создание нового столбца

Pandas позволяет выполнять операцию сразу над целыми столбцами:


In [ ]:
orders["total"] = orders["quantity"] * orders["unit_price"]

orders.head()


Это векторная операция — та же идея, которую мы видели в NumPy.

Мы не пишем цикл по каждой строке:


In [ ]:
# Не нужно:
#
# for order in orders:
#     ...


### Мини-практика 1

Создайте новый столбец `total_with_tax`, равный:

```text
total × 1.2
```

Сравните `total` и `total_with_tax`.


In [ ]:
# TODO


## 3. Изменяем исходный DataFrame или создаём копию?

Если написать:

```python
orders["total"] = ...
```

мы добавляем столбец прямо в существующий `orders`.

Если хотим экспериментировать, не меняя исходный объект:


In [ ]:
orders_copy = orders.copy()


Это полезная привычка перед крупными преобразованиями.

`copy()` создаёт отдельный объект DataFrame.


# Часть 2. Строковые данные

## 4. Строковые методы Pandas

В обычном Python:

```python
text.strip().lower()
```

В Pandas для строкового столбца используется `.str`:


In [ ]:
orders["category_clean"] = (
    orders["category"]
    .str.strip()
    .str.lower()
)

orders[["category", "category_clean"]].head(10)


Здесь:

```python
orders["category"]
```

— это `Series`.

А:

```python
.str.strip()
.str.lower()
```

— применение строковых операций ко всем значениям столбца.


### Частые строковые операции


In [ ]:
orders["product"].str.lower().head()


In [ ]:
orders["product"].str.contains("Книга").head()


In [ ]:
orders["product"].str.len().head()


### Мини-практика 2

1. Создайте столбец `product_lower`, где название товара записано в нижнем регистре.
2. Выберите только строки, где в `product` встречается слово `"Книга"`.
3. Создайте столбец `product_length` — длина названия товара.


In [ ]:
# TODO


# Часть 3. Даты

## 5. Почему дата сначала может быть строкой?

После чтения CSV Pandas часто воспринимает дату как обычный `object` / строку.


In [ ]:
print(orders["order_date"].dtype)


Для работы с датами преобразуем столбец:


In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

print(orders["order_date"].dtype)


Теперь Pandas понимает, что это дата, и появляется пространство имён `.dt`.


## 6. Извлечение частей даты


In [ ]:
orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.month
orders["day"] = orders["order_date"].dt.day
orders["weekday"] = orders["order_date"].dt.day_name()

orders[["order_date", "year", "month", "day", "weekday"]].head()


Можно фильтровать по датам так же, как по числам:


In [ ]:
orders[orders["order_date"] >= "2026-09-05"]


### Мини-практика 3

1. Создайте столбец `day_of_month`.
2. Создайте столбец `month`.
3. Выберите заказы, сделанные начиная с `2026-09-07`.
4. Посчитайте, сколько таких заказов.


In [ ]:
# TODO


# Часть 4. Группировки

## 7. Зачем `groupby`

До сих пор мы отвечали на вопросы вроде:

> Покажи только заказы из Москвы.

Теперь хотим вопросы другого типа:

> Какая средняя сумма заказа в каждой категории?

> Сколько заказов было в каждой категории?

> Какова общая выручка по категориям?

Для этого используется `groupby`.


Сначала оставим только оплаченные заказы:


In [ ]:
paid_orders = orders[orders["status"] == "paid"].copy()


## 8. Простая группировка

Средний `total` по категориям:


In [ ]:
paid_orders.groupby("category_clean")["total"].mean()


Общая сумма:


In [ ]:
paid_orders.groupby("category_clean")["total"].sum()


Количество строк:


In [ ]:
paid_orders.groupby("category_clean")["order_id"].count()


Логика:

```text
groupby("category_clean")
→ разбить строки на группы
→ ["total"]
→ выбрать показатель
→ mean() / sum() / count()
```


### Мини-практика 4

По оплачиваемым заказам получите для каждой категории:

1. средний `total`;
2. максимальный `total`;
3. количество заказов.


In [ ]:
# TODO


## 9. Несколько агрегатов сразу

Можно попросить несколько статистик:


In [ ]:
paid_orders.groupby("category_clean")["total"].agg(
    ["count", "sum", "mean", "max"]
)


Если нужны разные операции для разных столбцов:


In [ ]:
paid_orders.groupby("category_clean").agg({
    "total": ["sum", "mean"],
    "quantity": "sum",
})


На этом этапе достаточно понимать саму идею. Сложные схемы агрегирования не нужно заучивать.


### Мини-практика 5

Для каждой категории получите:

- сумму `total`;
- средний `total`;
- суммарное количество товаров `quantity`.


In [ ]:
# TODO


# Часть 5. Объединение таблиц

## 10. Зачем нужны две таблицы?

В `orders` есть:

```text
customer_id
```

Но нет города и сегмента клиента.

Они лежат в отдельной таблице:


In [ ]:
customers.head()


Связь между таблицами — поле:

```text
customer_id
```

Такое поле называют **ключом объединения**.


## 11. `merge`

Объединим заказы с информацией о клиентах:


In [ ]:
orders_with_customers = orders.merge(
    customers,
    on="customer_id",
    how="inner",
)

orders_with_customers.head()


Концептуально:

```text
orders.customer_id
        ↓
      merge
        ↑
customers.customer_id
```

Pandas сопоставляет строки по одинаковому значению ключа.


### Что такое `inner`

`inner` оставляет только строки, для которых ключ найден в обеих таблицах.

В наших заказах все `customer_id` существуют в таблице клиентов, поэтому все заказы сохраняются.


## 12. `left`

`left` сохраняет **все строки левой таблицы**.


In [ ]:
customers_with_orders = customers.merge(
    orders[["order_id", "customer_id", "total"]],
    on="customer_id",
    how="left",
)

customers_with_orders.tail()


В таблице `customers` есть клиент без заказов.

При `left` он всё равно остаётся, а данных заказа для него нет.

Это первый пример того, откуда в таблицах могут появляться пропуски. Подробно пропуски разберём отдельно.


### Мини-практика 6

1. Объедините `orders` и `customers` через `customer_id`.
2. Сохраните результат в `full_data`.
3. Оставьте столбцы:

```text
order_id
customer_name
city
segment
category_clean
total
status
```

4. Выведите первые 5 строк.


In [ ]:
# TODO


# Часть 6. Анализ после merge

Теперь в одной таблице есть и данные заказа, и данные клиента.


In [ ]:
full_data = orders.merge(
    customers,
    on="customer_id",
    how="inner",
)


## 13. Выручка по городам


In [ ]:
full_data[
    full_data["status"] == "paid"
].groupby("city")["total"].sum()


## 14. Средний чек по сегментам


In [ ]:
full_data[
    full_data["status"] == "paid"
].groupby("segment")["total"].mean()


### Мини-практика 7

По `full_data`:

1. посчитайте количество оплаченных заказов по городам;
2. найдите средний `total` по городам;
3. найдите общую сумму `total` по сегментам;
4. определите, в каком городе самый большой средний `total`.


In [ ]:
# TODO


# Итоговая практика

Используйте `orders` и `customers`.

1. Очистите `category` через `.str.strip().str.lower()`.
2. Создайте `total = quantity * unit_price`.
3. Преобразуйте `order_date` в дату.
4. Оставьте только оплаченные заказы с `2026-09-01`.
5. Объедините их с `customers`.
6. Получите таблицу со столбцами:

```text
order_id
order_date
customer_name
city
segment
category_clean
total
```

7. Посчитайте по каждому городу:
   - количество заказов;
   - общую сумму `total`;
   - средний `total`.

8. Коротко опишите в Markdown, какие этапы обработки вы выполнили.


In [ ]:
# TODO


# Что дальше

На следующем занятии логично перейти к **качеству данных**:

- пропуски;
- дубликаты;
- неверные типы;
- некорректные значения;
- базовые выбросы;
- проверки перед анализом.

То есть после сегодняшнего:

```text
преобразование данных
→ группировка
→ объединение таблиц
```

следующий шаг:

```text
проверка качества данных
→ очистка
→ подготовка к анализу и ML
```
